# Web后端开发 04：FastAPI工程结构

这一课解决一个很实际的问题：**接口越来越多以后，怎样把一个很长的 `app.py` 拆成多个职责清楚、仍然可以正常运行的文件？**

> 本课默认你已经学过：Python模块和包、函数、Pydantic数据模型、FastAPI路由、请求体、响应模型、状态码和TestClient。遇到相关概念时仍会快速复习。

## 1. 本课结束后，你应该能独立完成什么

完成本课后，你应该能够：

1. 解释为什么真实后端项目不适合把所有代码都塞进一个文件；
2. 分清 `main.py`、`schemas.py`、`services.py` 和 `routers/` 的职责；
3. 使用 `APIRouter` 定义一组路由，再用 `include_router()` 挂载到主应用；
4. 看懂包内相对导入，例如 `from ..schemas import ArticleCreate`；
5. 沿着“请求 → 路由 → 业务函数 → 响应”追踪一次完整调用；
6. 看懂 `python -m uvicorn article_demo.main:app --reload` 中每一部分；
7. 根据 404、导入错误和启动错误，判断工程结构的常见问题。

本课**不会**加入数据库、依赖注入、登录认证或异步数据库。这些概念互相独立，混在这一课里反而会干扰你掌握工程结构。

## 2. 运行本课前：安装必要依赖

请在 PowerShell 中进入本课目录：

```powershell
cd "02_Web后端开发/04_FastAPI工程结构"
python -m pip install -r requirements.txt
```

命令各部分的含义：

- `python`：使用当前环境中的 Python；
- `-m pip`：让这个 Python 运行它自己的 pip 模块，减少“装到了另一个环境”的问题；
- `install`：安装依赖；
- `-r requirements.txt`：读取文件中的依赖列表。`-r` 是关键字选项，后面跟文件路径。

也可以在当前 Notebook 内核中运行下一格的 `%pip`。二选一即可。`%pip` 是 Jupyter 魔法命令，会尽量把包装进当前内核使用的环境。安装后如果仍提示找不到包，请重启 Notebook 内核。

In [ ]:
# 如已在 PowerShell 安装，可跳过这一格。
%pip install -r requirements.txt

### 依赖分别做什么

- `fastapi`：Web框架，提供 `FastAPI`、`APIRouter` 和路由功能；
- `pydantic`：定义并验证请求、响应数据模型；
- `uvicorn[standard]`：运行 FastAPI 应用的 ASGI 服务器；`[standard]` 会附带常用性能和开发依赖；
- `httpx2`：当前环境中 `TestClient` 使用的进程内 HTTP 客户端依赖；安装包名和导入模块名都是 `httpx2`，不是旧模块名 `httpx`；
- `pytest`：运行作业自动测试。

下面的版本检查只负责确认导入成功。`__version__` 是模块提供的版本字符串；如果未来某个库不再公开它，也可以用 `python -m pip show 包名` 检查。

In [1]:
import fastapi
import httpx2
import pydantic
import pytest
import uvicorn

print("FastAPI:", fastapi.__version__)
print("Pydantic:", pydantic.__version__)
print("Uvicorn:", uvicorn.__version__)
print("httpx2:", httpx2.__version__)
print("pytest:", pytest.__version__)

FastAPI: 0.140.9
Pydantic: 2.10.3
Uvicorn: 0.51.0
httpx2: 2.9.1
pytest: 8.3.4


## 3. 为什么不能永远只用一个 `app.py`

一个只有两三个接口的程序，把模型、业务逻辑和路由写在一起很直观。但项目继续增长后，一个文件可能同时负责：

- 定义请求数据长什么样；
- 校验数据；
- 决定 URL 和 HTTP 方法；
- 计算或保存业务数据；
- 创建 FastAPI 主应用；
- 配置所有功能模块。

这就像把宿舍、教室、食堂和实验室全塞进一个房间：短期少走几步，长期很难找东西，也容易改坏无关功能。

**拆文件的核心不是追求文件多，而是让每个文件有清楚的职责。**

### 本课采用的四种职责

| 位置 | 负责什么 | 不应该主要负责什么 |
|---|---|---|
| `main.py` | 创建主应用、把各组路由装进应用 | 编写所有业务计算 |
| `schemas.py` | 用Pydantic描述请求和响应的数据形状 | 决定URL路径 |
| `services.py` | 业务处理，例如创建、查询、计算 | 决定HTTP状态码和接口路径 |
| `routers/articles.py` | 接收HTTP请求、调用业务函数、返回结果 | 保存全部业务规则 |

这不是 FastAPI 强制的唯一结构，而是适合当前阶段的一种清晰结构。以后项目更大时还可以继续拆分。

### 课堂小练习1：判断文件职责

**目标：**确认你能把代码放到合适的文件。

请补全 `file_roles` 字典。值只能从 `"应用入口"`、`"数据模型"`、`"业务逻辑"`、`"HTTP路由"` 中选择。正确时自动检测会显示通过。

In [2]:
file_roles = {
    "main.py": "应用入口",
    "schemas.py": "数据模型",
    "services.py": "业务逻辑",
    "routers/articles.py": "HTTP路由",
}

In [3]:
try:
    _check_expected = {
        "main.py": "应用入口",
        "schemas.py": "数据模型",
        "services.py": "业务逻辑",
        "routers/articles.py": "HTTP路由",
    }
    if "file_roles" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(file_roles, dict):
        print("❌ file_roles 必须是字典。")
    else:
        _check_wrong = [name for name, role in _check_expected.items() if file_roles.get(name) != role]
        if _check_wrong:
            print("❌ 这些文件的职责还不正确：", _check_wrong)
        else:
            print("✅ 小练习1通过：你已经能区分四类文件职责。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习1通过：你已经能区分四类文件职责。


<details>
<summary>完成后查看：为什么这样分</summary>

`main.py` 是应用入口；`schemas.py` 描述数据模型；`services.py` 放业务逻辑；`routers/articles.py` 负责HTTP路由。判断时不要只看文件名，也要看代码是否遵守了这个职责。

</details>

## 4. 先认清“包、模块和导入路径”

你在Python工程基础第02课学过模块和包，这里把它们放进真实FastAPI项目：

- 一个 `.py` 文件是一个**模块**，例如 `schemas.py`；
- 一个包含 Python 模块的目录可以作为**包**，例如 `article_demo/`；
- `__init__.py` 明确告诉阅读者和工具：这个目录是Python包；它可以是空文件；
- `routers/` 也是一个子包，所以里面也放了 `__init__.py`。

本课示例的实际结构：

```text
article_demo/
├── __init__.py
├── main.py
├── schemas.py
├── services.py
└── routers/
    ├── __init__.py
    └── articles.py
```

### 从文件路径换成模块路径

导入和 Uvicorn 启动命令使用的是**模块路径**，不是带 `.py` 的磁盘路径：

| 文件 | 模块路径 |
|---|---|
| `article_demo/main.py` | `article_demo.main` |
| `article_demo/schemas.py` | `article_demo.schemas` |
| `article_demo/routers/articles.py` | `article_demo.routers.articles` |

规则很直观：去掉 `.py`，再把目录分隔改成点。模块路径是否能被找到，还取决于你从哪个目录启动 Python。本课所有命令都要求在 `04_FastAPI工程结构` 目录执行。

### 相对导入中的一个点和两个点

在 `article_demo/services.py` 中：

```python
from .schemas import ArticleCreate
```

`.` 表示“当前包”，也就是 `article_demo`。

在 `article_demo/routers/articles.py` 中：

```python
from ..schemas import ArticleCreate
```

当前模块在 `article_demo.routers` 子包里，`..` 表示先回到上一层 `article_demo`，再找 `schemas`。

变化示例：`from ..services import create_article` 同样回到上一层，但导入的是 `services.py` 中的函数。

错误示例：把这里写成 `from .schemas ...` 会让Python尝试寻找 `article_demo.routers.schemas`，而这个模块不存在，所以产生 `ModuleNotFoundError`。

### 课堂小练习2：写出模块路径

**目标：**把文件位置转换成导入使用的模块路径。

请把 `article_demo/routers/articles.py` 对应的模块路径字符串赋给 `articles_module_path`。不要包含 `.py`，也不要使用 `/`。

In [6]:
articles_module_path = "article_demo.routers.articles"

In [7]:
try:
    if "articles_module_path" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(articles_module_path, str):
        print("❌ articles_module_path 必须是字符串。")
    elif articles_module_path == "article_demo/routers/articles.py":
        print("❌ 你写的是文件路径；模块路径要去掉 .py 并用点连接。")
    elif articles_module_path != "article_demo.routers.articles":
        print("❌ 模块路径还不正确，请重新对照转换规则。")
    else:
        print("✅ 小练习2通过：模块路径转换正确。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习2通过：模块路径转换正确。


<details><summary>完成后查看参考答案</summary>

模块路径是 `article_demo.routers.articles`。

</details>

### 让Notebook稳定找到本课包

VS Code有时把Notebook的当前工作目录设为仓库根目录，而示例包在更深一层。下面只做一次路径准备：

- `Path` 是标准库 `pathlib` 中表示文件系统路径的类；`Path(r"...")` 返回路径对象，字符串前的 `r` 让反斜杠按原样处理；
- `.is_dir()` 没有参数，检查路径是否为已存在的目录并返回布尔值；
- `sys.path` 是Python搜索可导入模块的目录列表；
- `sys.path.insert(0, value)` 是列表的 `insert` 方法：位置参数0表示插到开头，`value` 是要加入的目录字符串；它返回 `None`；
- 这个改动只影响当前Notebook内核，不会修改系统环境变量。

In [8]:
import sys
from pathlib import Path

lesson_dir = Path(".")
repo_relative_lesson_dir = Path("02_Web后端开发/04_FastAPI工程结构")
if repo_relative_lesson_dir.is_dir():
    lesson_dir = repo_relative_lesson_dir
if not lesson_dir.is_dir():
    raise FileNotFoundError(f"没有找到本课目录：{lesson_dir}")

lesson_dir_text = str(lesson_dir)
if lesson_dir_text not in sys.path:
    sys.path.insert(0, lesson_dir_text)

print("本课代码目录已加入模块搜索路径：", lesson_dir)

本课代码目录已加入模块搜索路径： 02_Web后端开发/04_FastAPI工程结构


## 5. 第一层：`schemas.py` 只描述数据

示例中的 `ArticleCreate` 描述客户端请求体，`ArticleResponse` 描述服务器响应体：

| 字段 | 类型 | 谁提供 | 用途 |
|---|---|---|---|
| `title` | 字符串 | 客户端必须提交 | 文章标题 |
| `content` | 字符串 | 客户端必须提交 | 文章正文 |
| `id` | 整数 | 服务器生成 | 唯一标识模拟值 |

这些名称都不是Python关键字，是我们根据业务自己定义的字段。`ArticleCreate` 不含 `id`，因为客户端创建文章时不知道服务器将分配什么编号；`ArticleResponse` 含 `id`，因为创建完成后服务器要把它返回。

`BaseModel` 和 `Field` 已在上一课详细学习。这里继续使用 `Field(..., min_length=..., max_length=...)`：`...` 表示没有默认值、字段必填；长度约束由Pydantic验证。

In [9]:
from pydantic import ValidationError
from article_demo.schemas import ArticleCreate, ArticleResponse

valid_article = ArticleCreate(title="模块化入门", content="把不同职责放进不同文件。")
print("正常数据：", valid_article)

try:
    ArticleCreate(title="", content="有正文但标题为空")
except ValidationError as error:
    print("错误示例：", error.errors()[0]["msg"])

正常数据： title='模块化入门' content='把不同职责放进不同文件。'
错误示例： String should have at least 1 character


## 6. 第二层：`services.py` 处理业务逻辑

“业务逻辑”就是系统真正要完成的业务规则。在示例中，它包括：

- 生成文章 `id`；
- 保存文章到内存列表；
- 返回全部文章。

它不需要知道请求来自 `/articles`，也不需要知道成功状态码是201。这种分离带来一个直接好处：不启动HTTP客户端，也能单独调用和测试业务函数。

### 本课示例中的服务函数

- `create_article(article)` 来自本课自己的 `article_demo.services` 模块；位置参数 `article` 是已经由Pydantic验证的 `ArticleCreate` 对象；返回 `ArticleResponse`；
- `list_articles()` 没有参数；返回 `list[ArticleResponse]`；
- `reset_articles()` 没有参数和有用返回值；它调用列表的 `.clear()` 清空演示数据，使重复运行Notebook时结果可预测；
- `_articles.copy()` 返回一个新列表。它里面仍是同一批文章对象，但调用者无法通过 `append()` 或 `clear()` 直接改变内部列表结构。

名字前的单下划线 `_articles` 是Python中的约定：表示它是模块内部实现细节，不建议其他模块直接操作。这不是强制权限控制。

In [10]:
from article_demo.services import create_article, list_articles, reset_articles

reset_articles()
created = create_article(
    ArticleCreate(title="第一篇", content="直接调用业务函数。")
)
print("创建结果：", created)
print("查询结果：", list_articles())

external_list = list_articles()
external_list.clear()
print("清空返回的新列表后，内部数据仍存在：", list_articles())

创建结果： id=1 title='第一篇' content='直接调用业务函数。'
查询结果： [ArticleResponse(id=1, title='第一篇', content='直接调用业务函数。')]
清空返回的新列表后，内部数据仍存在： [ArticleResponse(id=1, title='第一篇', content='直接调用业务函数。')]


### 课堂小练习3：只测试业务层

**目标：**体会业务函数不依赖HTTP。

1. 先调用 `reset_articles()`；
2. 创建两篇不同文章；
3. 把 `list_articles()` 的结果赋给变量 `service_result`。

自动检测只检查结果：必须有两篇文章，编号依次为1和2，标题依次为“包”和“路由”。

In [18]:
# 在这里完成三步操作
service_result = None
reset_articles()
create_article(
    ArticleCreate(title="包", content="我是第1篇")
)
create_article(
    ArticleCreate(title="路由", content="我是第2篇")
)
service_result = list_articles()
print(service_result)

[ArticleResponse(id=1, title='包', content='我是第1篇'), ArticleResponse(id=2, title='路由', content='我是第2篇')]


In [19]:
try:
    if "service_result" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(service_result, list):
        print("❌ service_result 应该是 list_articles() 返回的列表。")
    elif len(service_result) != 2:
        print("❌ service_result 应该恰好包含两篇文章。")
    elif [item.id for item in service_result] != [1, 2]:
        print("❌ 两篇文章的 id 应依次为1和2；请确认先清空再创建。")
    elif [item.title for item in service_result] != ["包", "路由"]:
        print("❌ 两篇文章的标题或创建顺序不符合题目。")
    else:
        print("✅ 小练习3通过：你已经能脱离HTTP单独调用业务层。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习3通过：你已经能脱离HTTP单独调用业务层。


<details><summary>完成后查看思路</summary>

依次调用一次 `reset_articles()`、两次 `create_article(ArticleCreate(...))`，最后调用 `list_articles()`。注意创建函数需要的是 `ArticleCreate` 对象，不是随意拼出的字典。

</details>

## 7. 第三层：`APIRouter` 管理一组HTTP路由

`APIRouter` 是 FastAPI 库提供的**类**。可以把它理解成“一小组可移动的路由”：先在功能模块中登记接口，稍后再把整组接口安装到主应用。它不是另一个独立运行的服务器。

本课使用的调用：

```python
router = APIRouter(prefix="/articles", tags=["articles"])
```

- `APIRouter` 来自 `fastapi`；调用后返回一个 `APIRouter` 对象；
- `prefix` 是关键字参数，给这一组路由统一添加路径前缀；这里是 `/articles`；
- `tags` 是关键字参数，接收字符串列表，在 `/docs` 和 OpenAPI 中给接口分类；它不改变路径。OpenAPI是描述接口路径、输入和输出的机器可读规范，FastAPI据此生成交互式文档；
- 这行在模块第一次被导入时执行，创建并配置 `router`。

### 路由装饰器仍然是以前学过的写法

```python
@router.post("", status_code=201, response_model=ArticleResponse)
def create_article_endpoint(article: ArticleCreate) -> ArticleResponse:
    return create_article(article)
```

这里首次变化只是把 `app.post` 换成 `router.post`：

- 第一个位置参数 `""` 是这条路由在共同前缀之后的路径；
- `status_code=201` 是关键字参数，规定成功响应的HTTP状态码；
- `response_model=ArticleResponse` 是关键字参数，要求FastAPI按响应模型验证和序列化返回值；“序列化”在这里就是把Python返回对象转换成可放进JSON响应体的数据；GET列表接口则使用 `response_model=list[ArticleResponse]`，表示响应是文章对象列表；
- 参数 `article: ArticleCreate` 触发FastAPI把JSON请求体解析、验证后交给函数；转换发生在框架内部，用户代码里没有单独的“JSON转模型”那一行；
- 路由函数只负责接住HTTP请求并转交给 `create_article()`。

边界示例：如果请求缺少 `title`，Pydantic验证失败，FastAPI自动返回422，路由函数和业务函数都不会执行。

### 最终路径如何计算

最终路径 = `APIRouter(prefix=...)` 的共同前缀 + 装饰器中的局部路径。

| `prefix` | 装饰器路径 | 最终路径 |
|---|---|---|
| `/articles` | `""` | `/articles` |
| `/articles` | `/recent` | `/articles/recent` |
| `/users` | `/{user_id}` | `/users/{user_id}` |

常见错误：`prefix="/articles"` 后又写 `@router.get("/articles")`，最终会变成 `/articles/articles`，不是 `/articles`。

### 课堂小练习4：计算最终路径

**目标：**区分共同前缀和局部路径。

已知 `prefix="/courses"`，三条装饰器路径依次是 `""`、`"/popular"`、`"/{course_id}"`。请把三个最终路径按顺序放进列表 `final_paths`。

In [20]:
final_paths = ["/courses", "/courses/popular", r"/courses/{course_id}"]

In [21]:
try:
    if "final_paths" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(final_paths, list):
        print("❌ final_paths 必须是列表。")
    elif final_paths == ["/courses", "/popular", "/{course_id}"]:
        print("❌ 后两项只写了局部路径，别忘了共同前缀。")
    elif final_paths != ["/courses", "/courses/popular", "/courses/{course_id}"]:
        print("❌ 路径或顺序不正确，请逐项做 prefix + 局部路径。")
    else:
        print("✅ 小练习4通过：三个最终路径都正确。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习4通过：三个最终路径都正确。


<details><summary>完成后查看参考答案</summary>

三个最终路径是 `/courses`、`/courses/popular`、`/courses/{course_id}`。`tags` 不参与路径计算。

</details>

## 8. 第四层：`main.py` 创建应用并挂载路由

`article_demo/main.py` 中最关键的三步是：

```python
from .routers.articles import router as articles_router
app = FastAPI(title="文章管理 API")
app.include_router(articles_router)
```

第一行中的 `as articles_router` 是导入别名：源模块里的变量仍叫 `router`，但在 `main.py` 中用更明确的名字 `articles_router`，以后同时导入用户路由、聊天路由时不容易混淆。

`include_router()` 是 `FastAPI` 对象的方法：

- 本课调用形式是 `app.include_router(articles_router)`；
- `articles_router` 是第一个位置参数，表示要加入主应用的 `APIRouter` 对象；
- 返回值是 `None`，它通过修改 `app` 内部的路由表完成工作；
- 这行在应用导入和初始化阶段执行，不是每次请求时执行。

如果只导入 `articles_router` 却忘了调用 `include_router()`，代码通常不会报语法错，但主应用根本没有 `/articles`，请求会得到404。

In [22]:
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient

mini_router = APIRouter(prefix="/demo")

@mini_router.get("")
def demo_endpoint() -> dict[str, str]:
    return {"message": "路由已经找到"}

app_without_router = FastAPI()
app_with_router = FastAPI()
app_with_router.include_router(mini_router)

# TestClient(app)：创建进程内测试客户端，app 是要测试的 FastAPI 对象。
without_client = TestClient(app_without_router)
with_client = TestClient(app_with_router)

print("未挂载：", without_client.get("/demo").status_code)
print("已挂载：", with_client.get("/demo").status_code)

未挂载： 404
已挂载： 200


上格的预期输出是“未挂载：404”和“已挂载：200”。注意：路由对象没有消失，它只是没有进入第一个主应用的路由表。

### 课堂小练习5：把路由挂载到应用

**目标：**亲手使用 `include_router()`。

下面已经创建了 `practice_router` 和 `/practice/check` 路由。请只补充必要代码，让 `practice_app` 包含这组路由。正确时检测请求会返回200和规定的JSON。

In [23]:
practice_router = APIRouter(prefix="/practice")

@practice_router.get("/check")
def practice_check() -> dict[str, bool]:
    return {"included": True}

practice_app = FastAPI()
# 在下方补充一行挂载代码
practice_app.include_router(practice_router)


In [24]:
try:
    if "practice_app" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(practice_app, FastAPI):
        print("❌ practice_app 必须是 FastAPI 应用对象。")
    else:
        _check_response = TestClient(practice_app).get("/practice/check")
        if _check_response.status_code == 404:
            print("❌ 主应用还没有这条路由，请确认调用了 include_router。")
        elif _check_response.status_code != 200:
            print("❌ 状态码应为200，实际为：", _check_response.status_code)
        elif _check_response.json() != {"included": True}:
            print("❌ 路由已找到，但响应体被意外修改了。")
        else:
            print("✅ 小练习5通过：路由已成功挂载到主应用。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习5通过：路由已成功挂载到主应用。


<details><summary>完成后查看思路</summary>

对 `practice_app` 调用 `include_router()`，把 `practice_router` 作为参数传入。这个方法会修改主应用的路由表。

</details>

## 9. 跑通真实的多文件示例

下面导入的是磁盘上的完整示例应用：

- `article_app` 是 `article_demo.main` 中创建并已经挂载路由的主应用；
- `article_client.post(path, json=...)` 模拟客户端发送POST请求；`path` 是位置参数，`json` 是关键字参数，字典会被编码为JSON请求体；
- `.json()` 把JSON响应体解码成Python数据并返回。

In [25]:
from article_demo.main import app as article_app
from article_demo.services import reset_articles

reset_articles()
article_client = TestClient(article_app)

create_response = article_client.post(
    "/articles",
    json={"title": "工程结构", "content": "每个文件负责一类事情。"},
)
print("创建状态码：", create_response.status_code)
print("创建响应：", create_response.json())

list_response = article_client.get("/articles")
print("查询状态码：", list_response.status_code)
print("查询响应：", list_response.json())

创建状态码： 201
创建响应： {'id': 1, 'title': '工程结构', 'content': '每个文件负责一类事情。'}
查询状态码： 200
查询响应： [{'id': 1, 'title': '工程结构', 'content': '每个文件负责一类事情。'}]


### 一次POST请求到底经过了什么

```text
客户端 POST /articles + JSON
        ↓
FastAPI 在主应用路由表中找到 articles_router 登记的路由
        ↓
FastAPI 把 JSON 解析并用 ArticleCreate 验证
        ↓  验证成功后才调用用户代码
routers/articles.py 的 create_article_endpoint(...)
        ↓
services.py 的 create_article(...) 生成id并保存
        ↓
路由函数返回 ArticleResponse
        ↓
FastAPI 按 response_model 验证、序列化并生成201响应
```

哪些是框架做的：匹配路由、解析JSON、调用Pydantic验证、生成HTTP响应。哪些是我们写的：文件职责、数据模型、路由声明、业务规则和返回值。

In [26]:
# 边界示例：空标题会在进入路由函数前被拒绝。
invalid_response = article_client.post(
    "/articles",
    json={"title": "", "content": "正文不为空"},
)
print(invalid_response.status_code)  # 预期422
print(invalid_response.json()["detail"][0]["loc"])

422
['body', 'title']


## 10. Uvicorn如何找到多文件应用

`FastAPI` 是框架和应用对象；`Uvicorn` 是真正监听网络端口、接收HTTP请求并把请求交给FastAPI的ASGI服务器。ASGI是一套规定“Python Web服务器怎样和异步Web应用通信”的接口标准；当前阶段不需要自己实现它，只要知道Uvicorn按这套标准调用FastAPI。只运行普通Python文件通常不会自动开始监听端口。

请在本课目录执行：

```powershell
python -m uvicorn article_demo.main:app --reload
```

逐段解释：

- `python -m uvicorn`：用当前Python运行Uvicorn模块；
- `article_demo.main`：模块路径，对应 `article_demo/main.py`；
- 冒号 `:`：把“模块路径”和“模块中的对象名”分开；
- `app`：`main.py` 中的变量名，也就是 `app = FastAPI(...)` 创建的对象；它不是固定单词。如果变量叫 `api`，命令就应写 `article_demo.main:api`；
- `--reload`：开发模式下监视代码文件，保存后自动重启服务器。它适合本地学习，不用于正式生产环境。

启动后可访问 `http://127.0.0.1:8000/docs`。终止服务器请在那个终端按 `Ctrl+C`。

### 启动目录为什么重要

Python要从当前目录等位置寻找顶层包 `article_demo`。如果你已经进入了 `article_demo/` 内部，却仍运行 `article_demo.main:app`，Python会尝试在里面再找一层 `article_demo`，容易出现 `ModuleNotFoundError`。

正确思路是：**站在能看见 `article_demo` 文件夹的父目录启动。** 本课就是 `04_FastAPI工程结构`。

### 课堂小练习6：拆解Uvicorn目标

**目标：**理解 `模块:对象`，而不是死记命令。

假设文件是 `shop_api/main.py`，其中主应用变量叫 `api`。请填写：

- `uvicorn_module`：模块路径；
- `uvicorn_object`：对象变量名；
- `uvicorn_target`：完整的 `模块:对象` 字符串。

In [27]:
uvicorn_module = "shop_api.main"
uvicorn_object = "api"
uvicorn_target = "shop_api.main:api"

In [28]:
try:
    _check_names = ["uvicorn_module", "uvicorn_object", "uvicorn_target"]
    _check_missing = [name for name in _check_names if name not in globals()]
    if _check_missing:
        print("❌ 请先运行答题单元格，缺少：", _check_missing)
    elif not all(isinstance(globals()[name], str) for name in _check_names):
        print("❌ 三个答案都必须是字符串。")
    elif uvicorn_module != "shop_api.main":
        print("❌ 模块路径不正确：去掉 .py，并把目录分隔改成点。")
    elif uvicorn_object != "api":
        print("❌ 对象名应与 main.py 中的主应用变量名一致。")
    elif uvicorn_target != "shop_api.main:api":
        print("❌ 完整目标应使用冒号连接模块路径和对象名。")
    else:
        print("✅ 小练习6通过：你已理解Uvicorn的模块:对象语法。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习6通过：你已理解Uvicorn的模块:对象语法。


<details><summary>完成后查看参考答案</summary>

模块路径是 `shop_api.main`，对象名是 `api`，完整目标是 `shop_api.main:api`。完整命令可以写成 `python -m uvicorn shop_api.main:api --reload`。

</details>

## 11. 常见错误：看到现象后先查哪里

| 现象 | 常见原因 | 最短检查方法 |
|---|---|---|
| `/health` 正常，但功能接口404 | 忘了 `include_router()`，或最终路径算错 | 打开 `/docs` 看接口是否存在，再检查前缀+局部路径 |
| `ModuleNotFoundError: No module named 'article_demo'` | 启动目录不对或包路径写错 | 确认当前目录能直接看见 `article_demo/` |
| `Could not import module ...` | `模块:对象` 的模块部分错误 | 对照磁盘路径去掉 `.py` 并改成点 |
| `Attribute ... not found` | 冒号后的应用变量名错误 | 打开 `main.py` 看 `FastAPI(...)` 赋给了哪个变量 |
| 实际路径出现两遍单词 | `prefix` 和装饰器路径重复 | 手工计算最终路径 |
| 导入到一半提示 partially initialized module | 两个模块互相在顶层导入，形成循环导入 | 画出导入箭头，调整职责和导入方向 |

**404不是“服务器没启动”。** 它表示请求已经到达一个服务器，但这个应用的路由表里没有匹配的方法和路径组合。

### 导入方向与循环导入

本课示例的导入方向是单向的：

```text
main → routers → services → schemas
          └──────────────→ schemas
```

`schemas` 不反过来导入 `main`，`services` 也不反过来导入 `routers`。如果A模块导入B，而B在初始化完成前又导入A，就可能形成循环导入。当前阶段最简单的预防方法是保持职责清楚，让依赖箭头大体朝一个方向。

### 课堂小练习7：排列请求调用顺序

**目标：**区分“应用启动时挂载路由”和“每次请求时执行的调用链”。

一次合法的 `POST /articles` 已经到达运行中的应用。请把下面四个字符串按执行先后放进 `request_flow`：

- `"响应模型序列化"`
- `"Pydantic验证请求体"`
- `"路由函数"`
- `"业务函数"`

注意：`include_router` 不放进去，因为它在应用初始化时已经完成。

In [31]:
request_flow = ["Pydantic验证请求体", "路由函数", "业务函数","响应模型序列化", ]

In [32]:
try:
    _check_expected = ["Pydantic验证请求体", "路由函数", "业务函数", "响应模型序列化"]
    if "request_flow" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(request_flow, list):
        print("❌ request_flow 必须是列表。")
    elif set(request_flow) != set(_check_expected):
        print("❌ 列表应恰好包含题目给出的四个阶段，不能缺少或增加。")
    elif request_flow != _check_expected:
        print("❌ 四个阶段都有了，但执行顺序还不正确。")
    else:
        print("✅ 小练习7通过：请求调用链顺序正确。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

✅ 小练习7通过：请求调用链顺序正确。


<details><summary>完成后查看参考答案与解释</summary>

先由Pydantic验证请求体，成功后FastAPI调用路由函数，路由函数再调用业务函数，返回值最后按响应模型序列化。验证失败时后面两个用户函数不会执行。

</details>

## 12. 完整结构回顾

同一个项目中存在两条容易混淆但不同的链：

**导入和初始化阶段（服务器启动时）：**

```text
Uvicorn导入 article_demo.main
→ main导入articles_router
→ 路由模块导入模型和业务函数
→ main创建app并include_router
→ 应用路由表准备完成
```

**请求处理阶段（每次请求时）：**

```text
请求到达 → 路由匹配与数据验证 → 路由函数 → 业务函数 → 响应验证与序列化
```

`include_router()` 属于第一条链，不会在每个请求中重复调用。

## 13. 课末作业：模块化学习任务 API

### 任务背景

你要完成一个保存在内存中的“学习任务 API”。重点不是数据库，而是把数据模型、业务逻辑、HTTP路由和应用入口放到正确文件，并让它们协作。

### 允许修改的文件

只修改以下四个文件中的 `TODO` 区域：

- `study_app/schemas.py`
- `study_app/services.py`
- `study_app/routers/tasks.py`
- `study_app/main.py`

不要修改 `test_study_app.py` 来降低要求，也不需要修改 `__init__.py`。

### 数据契约：字段到底由谁提供

`TaskCreate` 是客户端请求模型：

| 字段 | Python类型 | 是否必填/默认 | 约束 | 数据由谁提供 |
|---|---|---|---|---|
| `title` | `str` | 必填、无默认值 | 长度1至80 | 客户端必须提交 |
| `priority` | `int` | 非必填、默认1 | 1至5（含边界） | 客户端可提交；省略时Pydantic填1 |

`TaskResponse` 是服务器响应模型：

| 字段 | Python类型 | 响应值来自哪里 | 客户端创建时是否提交 |
|---|---|---|---|
| `id` | `int` | 服务器按创建顺序生成，从1开始 | 否 |
| `title` | `str` | 从已验证的请求数据取得 | 是 |
| `priority` | `int` | 从已验证的请求数据取得，可能是默认值 | 可选 |
| `completed` | `bool` | 服务器创建时固定生成 `False` | 否 |

`title`、`priority`、`completed` 都不是Python关键字，是本作业定义的业务字段。客户端不要提交 `id` 和 `completed`；它们由服务器负责生成。

### HTTP接口契约

主应用标题必须是 `学习任务 API`。任务路由组在自动文档中的标签必须是 `tasks`。

| 方法与最终路径 | 客户端输入 | 成功状态码 | 成功响应体 |
|---|---|---|---|
| `GET /health` | 无 | 200 | `{"status": "ok"}`（已提供） |
| `POST /tasks` | `TaskCreate` JSON请求体 | 201 | 新创建的完整 `TaskResponse` |
| `GET /tasks` | 无 | 200 | 按创建顺序排列的 `TaskResponse` 列表；初始为空列表 |

验证失败由FastAPI自动返回422，包括：缺少 `title`、空标题、`priority` 小于1或大于5。题目没有要求自定义422错误文本。

### 各文件必须实现的职责（这是验收契约，不是代码答案）

- `schemas.py`：定义上述两个Pydantic模型和字段约束；
- `services.py`：创建任务、生成递增 `id`、保存任务、查询任务；查询结果不能让外部通过修改列表结构直接破坏内部存储；
- `routers/tasks.py`：创建带共同前缀 `/tasks` 和标签 `tasks` 的路由组；定义POST和GET接口；路由函数把业务交给services；
- `main.py`：创建规定标题的主应用，并把任务路由组挂载进去；不要在这里复制一套 `/tasks` 路由。

测试只要求内存保存；程序重启后数据丢失是本课的预期行为。下一课学习数据库后才会解决持久化。

### 运行作业测试

在本课目录执行：

```powershell
python -m pytest test_study_app.py -v
```

应收集 **12个测试项**；全部完成后应为12 passed。一个参数化测试函数会分别使用优先级0和6，所以它生成两个测试项。

测试文件中你会看到：

- `TestClient(app, raise_server_exceptions=False)`：第二个关键字参数让暂时的服务器内部错误转成500响应，避免未完成时出现超长内部错误栈；
- `@pytest.fixture(autouse=True)`：每个测试前后自动清空内存列表，防止测试互相污染；
- `yield`：把fixture分成“测试前”和“测试后”两段；
- `@pytest.mark.parametrize(...)`：用多个输入重复运行同一测试；
- `app.openapi()`：让FastAPI返回描述接口文档的Python字典，测试用它确认 `/tasks` 已挂载且标签正确。

这些测试工具在Python工程基础pytest课程中已经学习；这里说明新出现的参数和具体用途。

### 手动启动作业应用

测试通过后，可以在本课目录运行：

```powershell
python -m uvicorn study_app.main:app --reload
```

打开 `http://127.0.0.1:8000/docs`，确认 `tasks` 分类下有GET和POST两个接口。先POST创建，再GET查询。

## 14. 理解题

请用自己的话回答。每道题都只检查前文已经讲解并演示过的内容。

1. `schemas.py` 和 `services.py` 分别负责什么？为什么不建议让 `schemas.py` 决定URL路径？
2. `APIRouter(prefix="/articles")` 配合 `@router.get("/recent")` 时，最终路径是什么？`tags` 会不会改变路径？
3. 只定义了 `articles_router`，却没有执行 `app.include_router(articles_router)`，请求 `/articles` 为什么会返回404？
4. `from ..schemas import ArticleCreate` 中两个点表示什么？如果误写成一个点，Python会尝试找哪个包里的 `schemas`？
5. 解释 `python -m uvicorn article_demo.main:app --reload` 中 `article_demo.main`、冒号后的 `app` 和 `--reload`。
6. 为什么一次请求中先做Pydantic请求验证，再执行路由函数和业务函数？验证失败时后两个函数是否会执行？

### 在这里填写理解题答案

1.
- schemas负责定义用户请求和服务器响应的格式，services负责编写具体的业务函数
- schemas.py 放的是数据模型；URL路径应该由 routers/tasks.py 决定，不是 main.py

2. 
- 最终路径是 `/articles/recent`
- tag不会改变路径，只会改变标签

3. 
- 因为没有把路由加入主应用的路由表

4. 
- .. 表示上级目录，如果只有一个.，python会尝试article_demo.routers.schemas
5. 
- `article_demo.main` 是模块路径，其中 `article_demo` 是包名，`main` 是模块名，对应 `main.py`；冒号后的 `app` 是 `main.py` 中保存FastAPI主应用对象的变量名；`--reload` 表示在开发模式下监视代码文件，保存后自动重启服务器。

6. 
- Pydantic先验证客户端请求体是否符合模型要求；如果不符合，FastAPI会直接返回422，路由函数和业务函数都不会执行。

## 15. 本课小结

你现在看到的不是“为了拆文件而拆文件”，而是一条明确协作链：

- `schemas` 规定数据形状；
- `services` 实现业务行为；
- `routers` 把HTTP请求转交给业务层；
- `main` 组装主应用；
- Uvicorn通过 `模块:对象` 找到并运行应用。

下一课学习数据库基础时，我们会把现在 `services.py` 中的内存列表替换成可持久保存的数据存储。工程结构先搭稳，后续替换数据层才不会把整个应用一起改乱。

## 可选参考资料

- [FastAPI官方教程：Bigger Applications - Multiple Files](https://fastapi.tiangolo.com/tutorial/bigger-applications/)
- [FastAPI官方教程：Testing](https://fastapi.tiangolo.com/tutorial/testing/)

官方教程适合在完成本课后对照阅读；第一次学习仍建议按本Notebook的顺序逐格运行。